<a href="https://colab.research.google.com/github/supriyag123/PHD_Pub/blob/main/AGENTIC-MODULE6.3-Ablation%20Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# MODULE 6.3 — ASPIRE ABLATION STUDY
# Standalone notebook — no X, no Transformer, no sensor models
# All results derived from PRECOMP[:2000] matching paper evaluation
# ============================================================

import numpy as np
import json
import os
from sklearn.metrics import precision_score, recall_score, f1_score
from typing import Optional, Dict, Any, List
from datetime import datetime

# ============================================================
# PATHS
# ============================================================
PRECOMP_PATH  = "/content/drive/MyDrive/PHD/2025/cache/holdout_precomp.json"
label_path    = "/content/drive/MyDrive/PHD/2025/TEMP_OUTPUT_METROPM/window_labels_3class.npy"
holdout_mask_path = "/content/drive/MyDrive/PHD/2025/TEMP_OUTPUT_METROPM/holdout_mask.npy"
ABLATION_PATH = "/content/drive/MyDrive/PHD/2025/ablation_results.json"

# ============================================================
# PAPER EVALUATION WINDOW — must match original paper run
# ============================================================
N_PAPER          = 2000
MARGIN_THR       = 0.0338
WARN_DEFICIT_THR = 0.03

BASE_POLICY = dict(
    w_sensor=0.20,
    w_window=0.80,
    warn_threshold=0.50,
    failure_threshold=0.50,
    drift_threshold=0.35,
)

# ============================================================
# LOAD — labels and PRECOMP only
# ============================================================
print("Loading labels...")
y            = np.load(label_path)
holdout_mask = np.load(holdout_mask_path).astype(bool)
y_hold       = y[holdout_mask]
print(f"✅ y_hold shape: {y_hold.shape}")

print("Loading PRECOMP (~30 seconds)...")
with open(PRECOMP_PATH, "r") as f:
    PRECOMP = json.load(f)
print(f"✅ PRECOMP loaded: {len(PRECOMP)} total samples")

# Slice to paper evaluation window
PRECOMP_2k = PRECOMP[:N_PAPER]
y_hold_2k  = y_hold[:N_PAPER]
print(f"✅ Using first {N_PAPER} samples — matching paper evaluation")

# Quick sanity check — confirm y_true in PRECOMP matches y_hold
mismatch = sum(
    1 for i, row in enumerate(PRECOMP_2k)
    if int(row["y_true"]) != int(y_hold_2k[i])
)
if mismatch == 0:
    print("✅ PRECOMP y_true matches y_hold — data aligned correctly")
else:
    print(f"⚠️ WARNING: {mismatch} y_true mismatches — check data alignment")

# ============================================================
# DECISION AGENT — minimal self-contained version
# ============================================================

class DecisionAgent:

    def __init__(
        self,
        w_sensor: float = 0.20,
        w_window: float = 0.80,
        w_wdi: float = 0.60,
        w_sensor_drift: float = 0.30,
        w_fdi: float = 0.10,
        failure_threshold: float = 0.50,
        failure_critical_threshold: float = 0.80,
        detection_threshold: float = 0.50,
        drift_threshold: float = 0.35,
        warn_threshold: float = 0.50,
        warn_gate_high_contra: float = 0.65,
        warn_boost: float = 0.15,
        warn_penalty: float = 0.25,
        alert_low: float = 0.35,
        alert_med: float = 0.55,
        alert_high: float = 0.75,
        topk_sensors: int = 5,
        use_confidence: bool = True,
    ):
        self.w_sensor                  = w_sensor
        self.w_window                  = w_window
        self.w_wdi                     = w_wdi
        self.w_sensor_drift            = w_sensor_drift
        self.w_fdi                     = w_fdi
        self.failure_threshold         = failure_threshold
        self.failure_critical_threshold = failure_critical_threshold
        self.detection_threshold       = detection_threshold
        self.drift_threshold           = drift_threshold
        self.warn_threshold            = warn_threshold
        self.warn_gate_high_contra     = warn_gate_high_contra
        self.warn_boost                = warn_boost
        self.warn_penalty              = warn_penalty
        self.alert_low                 = alert_low
        self.alert_med                 = alert_med
        self.alert_high                = alert_high
        self.topk_sensors              = topk_sensors
        self.use_confidence            = use_confidence

    def _clip01(self, x):
        try:
            return float(np.clip(float(x), 0.0, 1.0))
        except Exception:
            return 0.0

    def _sensor_intensity(self, master_output):
        if not master_output:
            return {
                "sensor_anom_intensity":   0.0,
                "sensor_drift_intensity":  0.0,
                "sensor_retrain_intensity": 0.0,
                "top_sensors": [],
                "rates": {
                    "anomaly_rate": 0.0,
                    "drift_rate":   0.0,
                    "retrain_rate": 0.0,
                }
            }
        sys_dec = master_output.get("system_decisions", {}) or {}
        rates = {
            "anomaly_rate": float(sys_dec.get("anomaly_rate", 0.0)),
            "drift_rate":   float(sys_dec.get("drift_rate",   0.0)),
            "retrain_rate": float(sys_dec.get("retrain_rate", 0.0)),
        }
        sensor_results = master_output.get("sensor_results", []) or []
        strengths = []
        for r in sensor_results:
            score  = float(r.get("anomaly_score", 0.0))
            conf   = float(r.get("confidence", 0.5))
            is_an  = 1.0 if r.get("is_anomaly", False) else 0.0
            dr     = 1.0 if r.get("drift_flag", False) else 0.0
            rt     = 1.0 if r.get("needs_retrain_flag", False) else 0.0
            base   = score * (conf if self.use_confidence else 1.0)
            strengths.append({
                "strength":          float(base + 0.25 * is_an),
                "drift_flag":        bool(dr),
                "needs_retrain_flag": bool(rt),
            })
        strengths_sorted = sorted(
            strengths, key=lambda x: x["strength"], reverse=True
        )
        top  = strengths_sorted[:max(1, self.topk_sensors)]
        vals = np.array([t["strength"] for t in top], dtype=float)
        top_strength_mean = float(
            1.0 - np.exp(-np.mean(np.maximum(vals, 0.0)))
        ) if len(vals) else 0.0
        drift_flags   = np.array(
            [1.0 if t["drift_flag"] else 0.0 for t in strengths_sorted],
            dtype=float
        )
        retrain_flags = np.array(
            [1.0 if t["needs_retrain_flag"] else 0.0 for t in strengths_sorted],
            dtype=float
        )
        return {
            "sensor_anom_intensity": self._clip01(
                0.5 * rates["anomaly_rate"] + 0.5 * top_strength_mean),
            "sensor_drift_intensity": self._clip01(
                0.6 * rates["drift_rate"] +
                0.4 * float(drift_flags.mean() if len(drift_flags) else 0.0)),
            "sensor_retrain_intensity": self._clip01(
                0.6 * rates["retrain_rate"] +
                0.4 * float(retrain_flags.mean() if len(retrain_flags) else 0.0)),
            "top_sensors": top,
            "rates": rates,
        }

    def _window_intensity(self, window_output):
        if not window_output:
            return {
                "win_anom_intensity":  0.0,
                "win_drift_intensity": 0.0,
                "fds": 0.0, "fdi": 0.0, "wss": 0.0, "wdi": 0.0,
                "event_type": None, "severity": 0.0,
                "window_mse": None, "predicted_window": None,
            }
        event_type = window_output.get("event_type")
        severity   = float(window_output.get("severity", 0.0) or 0.0)
        fds = float(window_output.get("fds", 0.0) or 0.0)
        fdi = float(window_output.get("fdi", 0.0) or 0.0)
        wss = float(window_output.get("wss", 0.0) or 0.0)
        wdi = float(window_output.get("wdi", 0.0) or 0.0)
        window_mse = None
        if window_output.get("forecast_metrics"):
            window_mse = window_output["forecast_metrics"].get("mse")
        wss_int = 1.0 / (1.0 + np.exp(-abs(wss)))
        fds_int = 1.0 / (1.0 + np.exp(-fds))
        sev_int = 1.0 - np.exp(-max(severity, 0.0))
        win_anom = self._clip01(
            0.85 * wss_int + 0.10 * sev_int + 0.05 * fds_int +
            (0.05 if event_type == "ANOMALY" else 0.0)
        )
        win_drift = self._clip01(
            0.85 * wdi + 0.15 * fdi +
            (0.10 if event_type == "DRIFT" else 0.0)
        )
        return {
            "win_anom_intensity":  float(win_anom),
            "win_drift_intensity": float(win_drift),
            "fds": fds, "fdi": fdi, "wss": wss, "wdi": wdi,
            "event_type":      event_type,
            "severity":        severity,
            "window_mse":      float(window_mse) if window_mse is not None else None,
            "predicted_window": window_output.get("predicted_window"),
        }

    def _prediction_probs(self, model_outputs):
        if not model_outputs:
            return {"p_fault": 0.0, "p_warn": 0.0, "p_normal": 1.0}
        p_fault = float(model_outputs.get("failure_prob",     0.0) or 0.0)
        p_warn  = float(model_outputs.get("transformer_prob", 0.0) or 0.0)
        p_norm  = float(model_outputs.get(
            "p_normal", max(0.0, 1.0 - p_fault - p_warn)))
        return {
            "p_fault":  self._clip01(p_fault),
            "p_warn":   self._clip01(p_warn),
            "p_normal": self._clip01(p_norm),
        }

    def decide(self, master_output, window_output,
               model_outputs=None, metadata=None):

        sens = self._sensor_intensity(master_output)
        win  = self._window_intensity(window_output)
        pred = self._prediction_probs(model_outputs)

        detection_risk = self._clip01(
            self.w_sensor * sens["sensor_anom_intensity"] +
            self.w_window * win["win_anom_intensity"]
        )
        drift_risk = self._clip01(
            max(win["wdi"], sens["sensor_drift_intensity"])
        )

        final_failure = (
            pred["p_fault"] >= self.failure_threshold
            or (
                pred["p_warn"] >= self.warn_threshold
                and detection_risk >= 0.65
                and drift_risk    >= 0.60
            )
        )

        final_anomaly = detection_risk >= self.detection_threshold
        final_drift   = drift_risk     >= self.drift_threshold

        arg         = (model_outputs.get("transformer_argmax")
                       if model_outputs else None)
        warn_margin = float(pred["p_warn"] - pred["p_normal"])

        if arg == 1:
            if warn_margin >= 0.06:
                warning_level = "HIGH"
            elif warn_margin >= 0.04:
                warning_level = "MEDIUM"
            else:
                warning_level = "LOW"
        else:
            warning_level = None

        margin_gate         = warn_margin >= MARGIN_THR
        transformer_warning = (arg == 1) and margin_gate
        near_miss           = (arg == 0) and (
            0 < (pred["p_normal"] - pred["p_warn"]) < WARN_DEFICIT_THR
        )
        warning_evidence    = near_miss and (
            drift_risk     >= self.drift_threshold or
            detection_risk >= self.detection_threshold
        )

        final_warning = (
            (transformer_warning or warning_evidence)
            and not final_failure
        )

        return {
            "final_warning": bool(final_warning),
            "final_failure": bool(final_failure),
            "final_anomaly": bool(final_anomaly),
            "final_drift":   bool(final_drift),
            "scores": {
                "p_warn":          pred["p_warn"],
                "p_fault":         pred["p_fault"],
                "p_normal":        pred["p_normal"],
                "warn_margin":     float(warn_margin),
                "warning_level":   warning_level,
                "detection_risk":  float(detection_risk),
                "drift_risk":      float(drift_risk),
            },
        }


# ============================================================
# ABLATION RUNNER
# ============================================================

def run_ablation_variant(precomp, y_hold, variant_name,
                          use_sensor=True,
                          use_window=True,
                          margin_thr=MARGIN_THR,
                          warn_deficit=WARN_DEFICIT_THR,
                          policy=None):
    if policy is None:
        policy = BASE_POLICY.copy()

    agent       = DecisionAgent(**policy)
    dec_warn    = []
    y_warn_true = []
    y_fail_true = []

    for i, row in enumerate(precomp):
        y_true = int(row["y_true"])
        mo     = row.get("model_outputs", {}) or {}

        p0     = float(mo.get("p_normal", 0.0))
        p1     = float(mo.get("p_warn",   0.0))
        p2     = float(mo.get("p_fault",  0.0))
        argmax = int(mo.get("argmax", np.argmax([p0, p1, p2])))

        master_out = row["master"] if use_sensor else None
        window_out = row["window"] if use_window else None

        decision = agent.decide(
            master_output=master_out,
            window_output=window_out,
            model_outputs={
                "failure_prob":       p2,
                "transformer_prob":   p1,
                "transformer_argmax": argmax,
                "p_normal":           p0,
            },
            metadata={"index": i}
        )

        warn_margin = p1 - p0

        if not use_sensor and not use_window:
            # V6: post-processing only — margin gate, no context
            warn = int((argmax == 1) and (warn_margin >= margin_thr))

        elif margin_thr == 0.0:
            # V4: no margin gate
            warn = int((argmax == 1) and not decision["final_failure"])

        elif warn_deficit == 0.0:
            # V5: no near-miss recovery
            warn = int(
                (argmax == 1)
                and (warn_margin >= margin_thr)
                and not decision["final_failure"]
            )

        else:
            # V1, V2, V3: full decision agent output
            warn = int(decision["final_warning"])

        dec_warn.append(warn)
        y_warn_true.append(int(y_true == 1))
        y_fail_true.append(int(y_true == 2))

    y_w = np.array(y_warn_true)
    d_w = np.array(dec_warn)

    return {
        "variant":           variant_name,
        "warning_precision": float(precision_score(y_w, d_w, zero_division=0)),
        "warning_recall":    float(recall_score(y_w,    d_w, zero_division=0)),
        "warning_f1":        float(f1_score(y_w,        d_w, zero_division=0)),
        "n_samples":         len(y_w),
    }


# ============================================================
# V0 — Transformer argmax baseline
# Derived directly from PRECOMP, not from state
# ============================================================
y_warn_2k = np.array([int(r["y_true"] == 1) for r in PRECOMP_2k])
y_fail_2k = np.array([int(r["y_true"] == 2) for r in PRECOMP_2k])
tf_warn_2k = np.array([
    int(r["model_outputs"]["argmax"] == 1) for r in PRECOMP_2k
])

v0 = {
    "variant":           "V0: Transformer argmax (baseline)",
    "warning_precision": float(precision_score(y_warn_2k, tf_warn_2k,
                                               zero_division=0)),
    "warning_recall":    float(recall_score(y_warn_2k,    tf_warn_2k,
                                            zero_division=0)),
    "warning_f1":        float(f1_score(y_warn_2k,        tf_warn_2k,
                                        zero_division=0)),
    "n_samples":         N_PAPER,
}
print(f"✅ V0: P={v0['warning_precision']:.3f}  "
      f"R={v0['warning_recall']:.3f}  F1={v0['warning_f1']:.3f}")
print(f"   (expected: P=0.875  R=0.941  F1=0.906)")


# ============================================================
# V1–V6 — all variants
# ============================================================
print("\nRunning ablation variants (each ~1 min)...")

v1 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V1: Full ASPIRE (reported result)")
print(f"✅ V1: P={v1['warning_precision']:.3f}  "
      f"R={v1['warning_recall']:.3f}  F1={v1['warning_f1']:.3f}")
print(f"   (expected: P=0.889  R=0.952  F1=0.919)")

v2 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V2: No sensor agents",
    use_sensor=False)
print(f"✅ V2 done: P={v2['warning_precision']:.3f}  "
      f"R={v2['warning_recall']:.3f}  F1={v2['warning_f1']:.3f}")

v3 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V3: No window agent",
    use_window=False)
print(f"✅ V3 done: P={v3['warning_precision']:.3f}  "
      f"R={v3['warning_recall']:.3f}  F1={v3['warning_f1']:.3f}")

v4 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V4: No margin gate",
    margin_thr=0.0)
print(f"✅ V4 done: P={v4['warning_precision']:.3f}  "
      f"R={v4['warning_recall']:.3f}  F1={v4['warning_f1']:.3f}")

v5 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V5: No near-miss recovery",
    warn_deficit=0.0)
print(f"✅ V5 done: P={v5['warning_precision']:.3f}  "
      f"R={v5['warning_recall']:.3f}  F1={v5['warning_f1']:.3f}")

v6 = run_ablation_variant(
    PRECOMP_2k, y_hold_2k,
    "V6: Post-proc baseline (margin gate only)",
    use_sensor=False,
    use_window=False,
    margin_thr=MARGIN_THR)
print(f"✅ V6 done: P={v6['warning_precision']:.3f}  "
      f"R={v6['warning_recall']:.3f}  F1={v6['warning_f1']:.3f}")


# ============================================================
# RESULTS TABLE
# ============================================================
results = [v0, v1, v2, v3, v4, v5, v6]

print(f"\n{'Variant':<50} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 80)
for r in results:
    marker = "  ◀ reported" if "V1" in r["variant"] else ""
    print(f"{r['variant']:<50} "
          f"{r['warning_precision']:>10.3f} "
          f"{r['warning_recall']:>8.3f} "
          f"{r['warning_f1']:>8.3f}{marker}")


# ============================================================
# SAVE TO DRIVE
# ============================================================
with open(ABLATION_PATH, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n✅ Saved to {ABLATION_PATH}")

Loading labels and masks...
✅ y_hold: (346681,)
Loading checkpoint state...
✅ State loaded — 50000 samples evaluated
Loading PRECOMP (this takes ~30 seconds)...
✅ PRECOMP loaded — 72250 cached samples
✅ V0 done
Running V1–V6 (each pass over PRECOMP takes ~1–2 min)...
✅ V1 done
✅ V2 done
✅ V3 done
✅ V4 done
✅ V5 done
✅ V6 done

Variant                                           Precision   Recall       F1
------------------------------------------------------------------------------
V0: Transformer argmax (baseline)                     0.688    0.941    0.795
V1: Full ASPIRE (reported result)                     0.402    0.953    0.565  ◀ reported
V2: No sensor agents                                  0.389    0.953    0.553
V3: No window agent                                   0.307    0.905    0.459
V4: No margin gate                                    0.395    0.941    0.556
V5: No near-miss recovery                             0.400    0.917    0.557
V6: Post-proc baseline (margin gat